# Delivery Delay Prediction — Local Hyperparameter Tuning (Optuna)

The OML Notebook environment has no internet access (its pip index points at an
internal Oracle artifactory host that isn't reachable from this ADB's network), so
`optuna`/`xgboost`/`lightgbm` can't be installed there. Doing the tuning locally
instead, using the same `db/connection.py` / Vault credential pattern as the rest of
this project. The Oracle side of this project is still the data platform (the
`DELIVERY_DELAY_FEATURES` table, built and reviewed properly) -- this notebook just
runs the actual training/tuning code locally rather than inside OML4Py.

**Same framing as `ml/delivery_delay/delivery_delay_oml4py.ipynb`:** checkout-time
prediction, target = `DELIVERY_DELAY_DAYS` (Yeo-Johnson transformed for training),
same 80/20 time-based split, same VIF-driven feature list (`PRIMARY_PRODUCT_CATEGORY_NAME`
and `AVG_DISTANCE_KM` dropped) -- so results are directly comparable to the clean GLM
baseline (`R^2 = 0.418`, `RMSE` on the transformed scale `= sqrt(0.582889) = 0.7635`).

## 1. Pull the feature table, recreate the same time-based split

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # repo root, for db.connection

import pandas as pd
from db.connection import get_connection

conn = get_connection()
df = pd.read_sql("SELECT * FROM DELIVERY_DELAY_FEATURES", conn)
print(df.shape)
df.head()

/tmp/ipykernel_278253/507449488.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM DELIVERY_DELAY_FEATURES", conn)


(96470, 23)


,ORDER_ID,ORDER_PURCHASE_TIMESTAMP,ORDER_ESTIMATED_DELIVERY_DATE,ORDER_DELIVERED_CUSTOMER_DATE,PROMISED_DELIVERY_DAYS,DELIVERY_DELAY_DAYS,NUM_ITEMS,TOTAL_PRICE,TOTAL_FREIGHT_VALUE,TOTAL_WEIGHT_G,...,PRIMARY_SELLER_ID,PRIMARY_SELLER_STATE,HEAVIEST_PRODUCT_WEIGHT_G,HEAVIEST_PRODUCT_LENGTH_CM,HEAVIEST_PRODUCT_HEIGHT_CM,HEAVIEST_PRODUCT_WIDTH_CM,CUSTOMER_STATE,SAME_STATE_FLAG,MAX_DISTANCE_KM,AVG_DISTANCE_KM
0,de937fa64e3c6113c5a4c20ec533674c,2017-08-09 20:46:56,2017-08-31,2017-08-21 16:52:00,21.134074,-9.297222,1,129.00,32.23,8875.0,...,001cca7ae9ae17fb1caed9dfb1094831,ES,8875.0,40.0,14.0,43.0,RJ,0,417.261650,417.261650
1,dc3f947c4795561fca218d2ae362a4ef,2017-11-13 11:36:48,2017-12-08,2017-11-27 15:03:33,24.516111,-10.372535,1,195.99,86.53,4200.0,...,004c9cd9d87a3c30c522c48c4fc07416,SP,4200.0,64.0,40.0,48.0,RJ,0,595.743283,595.743283
2,de96b93d26af6d6a5c3aa622e245b86f,2017-02-02 12:34:47,2017-03-07,2017-02-06 14:55:28,32.475845,-28.378148,1,109.99,14.54,1800.0,...,004c9cd9d87a3c30c522c48c4fc07416,SP,1800.0,30.0,10.0,38.0,SP,1,351.985416,351.985416
3,dd3e21a337aa1b10edf9ce57c0772a2e,2018-03-07 16:43:19,2018-04-05,2018-03-20 23:12:52,28.303252,-15.032731,1,85.00,13.71,321.0,...,00fc707aaaad2d31347cf883cd2dfe10,PR,321.0,19.0,14.0,13.0,SP,0,557.783815,557.783815
4,dcbca116084725bc77c3e8b6480c4d82,2017-08-22 17:12:58,2017-09-18,2017-08-24 21:05:22,26.282662,-24.121273,3,162.00,47.46,850.0,...,013900e863eace745d3ec7614cab5b1a,PR,300.0,17.0,7.0,24.0,RJ,0,667.204983,667.204983


In [2]:
# Same time-based split as the OML4Py notebook -- train on earlier orders, test on
# the most recent ones. Must match exactly for the comparison against GLM to be fair.
df = df.sort_values('ORDER_PURCHASE_TIMESTAMP').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f'Train: {len(train_df)} orders, {train_df["ORDER_PURCHASE_TIMESTAMP"].min()} to {train_df["ORDER_PURCHASE_TIMESTAMP"].max()}')
print(f'Test:  {len(test_df)} orders, {test_df["ORDER_PURCHASE_TIMESTAMP"].min()} to {test_df["ORDER_PURCHASE_TIMESTAMP"].max()}')

Train: 77176 orders, 2016-09-15 12:16:38 to 2018-05-26 18:16:57
Test:  19294 orders, 2018-05-26 18:18:03 to 2018-08-29 15:00:37


**Check:** should match the OML4Py notebook exactly -- 77,176 train / 19,294 test,
boundary at 2018-05-26.

## 2. Yeo-Johnson on the target (fit on train only), encode categoricals, carve a validation split

In [3]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='yeo-johnson')
train_df['DELAY_TRANSFORMED'] = pt.fit_transform(train_df[['DELIVERY_DELAY_DAYS']])
test_df['DELAY_TRANSFORMED'] = pt.transform(test_df[['DELIVERY_DELAY_DAYS']])

# Same VIF-driven feature list as the clean GLM baseline (dropped PRIMARY_PRODUCT_CATEGORY_NAME
# and AVG_DISTANCE_KM -- see ml-engineer review in the OML4Py notebook)
exclude_cols = [
    'ORDER_ID', 'ORDER_PURCHASE_TIMESTAMP',
    'ORDER_DELIVERED_CUSTOMER_DATE', 'ORDER_ESTIMATED_DELIVERY_DATE',
    'PRIMARY_PRODUCT_ID', 'PRIMARY_SELLER_ID',
    'PRIMARY_PRODUCT_CATEGORY_NAME', 'AVG_DISTANCE_KM',
    'DELIVERY_DELAY_DAYS', 'DELAY_TRANSFORMED',
]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
categorical_cols = ['PRIMARY_PRODUCT_CATEGORY_NAME_ENGLISH', 'PRIMARY_SELLER_STATE', 'CUSTOMER_STATE']
print('Features:', feature_cols)
print('Categorical:', categorical_cols)

# XGBoost/LightGBM both support pandas 'category' dtype natively -- no one-hot needed
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

X_train_full, y_train_full = train_df[feature_cols], train_df['DELAY_TRANSFORMED']
X_test, y_test = test_df[feature_cols], test_df['DELAY_TRANSFORMED']

# Validation carve-out for Optuna scoring -- time-based (not random), and test_df stays
# a completely untouched final holdout, never seen during tuning
val_split_idx = int(len(train_df) * 0.85)
X_tr, y_tr = X_train_full.iloc[:val_split_idx], y_train_full.iloc[:val_split_idx]
X_val, y_val = X_train_full.iloc[val_split_idx:], y_train_full.iloc[val_split_idx:]
print(f'Tuning train: {len(X_tr)}, tuning val: {len(X_val)}, final test (untouched): {len(X_test)}')

Features: ['PROMISED_DELIVERY_DAYS', 'NUM_ITEMS', 'TOTAL_PRICE', 'TOTAL_FREIGHT_VALUE', 'TOTAL_WEIGHT_G', 'PRIMARY_PRODUCT_CATEGORY_NAME_ENGLISH', 'PRIMARY_SELLER_STATE', 'HEAVIEST_PRODUCT_WEIGHT_G', 'HEAVIEST_PRODUCT_LENGTH_CM', 'HEAVIEST_PRODUCT_HEIGHT_CM', 'HEAVIEST_PRODUCT_WIDTH_CM', 'CUSTOMER_STATE', 'SAME_STATE_FLAG', 'MAX_DISTANCE_KM']
Categorical: ['PRIMARY_PRODUCT_CATEGORY_NAME_ENGLISH', 'PRIMARY_SELLER_STATE', 'CUSTOMER_STATE']
Tuning train: 65599, tuning val: 11577, final test (untouched): 19294


## 3. Optuna + XGBoost

In [4]:
import optuna
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'enable_categorical': True,
        'tree_method': 'hist',
        'random_state': 42,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return mean_squared_error(y_val, preds)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('Best val MSE:', study.best_value)
print('Best params:', study.best_params)

[I 2026-07-14 15:28:36,917] A new study created in memory with name: no-name-1cb6bf01-08fb-4155-8ff7-4553438b2242


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-14 15:28:41,650] Trial 0 finished with value: 0.5517514980690404 and parameters: {'n_estimators': 437, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'reg_alpha': 2.5348407664333426e-07, 'reg_lambda': 3.3323645788192616e-08, 'min_child_weight': 18}. Best is trial 0 with value: 0.5517514980690404.


[I 2026-07-14 15:28:45,997] Trial 1 finished with value: 0.4734775592688011 and parameters: {'n_estimators': 641, 'max_depth': 8, 'learning_rate': 0.010725209743171997, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'reg_alpha': 8.148018307012941e-07, 'reg_lambda': 4.329370014459266e-07, 'min_child_weight': 4}. Best is trial 1 with value: 0.4734775592688011.


[I 2026-07-14 15:28:48,162] Trial 2 finished with value: 0.4752495832193773 and parameters: {'n_estimators': 374, 'max_depth': 7, 'learning_rate': 0.04345454109729477, 'subsample': 0.645614570099021, 'colsample_bytree': 0.8059264473611898, 'reg_alpha': 1.8007140198129195e-07, 'reg_lambda': 4.258943089524393e-06, 'min_child_weight': 8}. Best is trial 1 with value: 0.4734775592688011.


...

[I 2026-07-14 15:30:13,302] Trial 45 finished with value: 0.4789050296305895 and parameters: {'n_estimators': 252, 'max_depth': 3, 'learning_rate': 0.02763791616749076, 'subsample': 0.6584860940391833, 'colsample_bytree': 0.5346995686798042, 'reg_alpha': 0.1724055795199568, 'reg_lambda': 1.3589568771552378e-08, 'min_child_weight': 2}. Best is trial 33 with value: 0.46130134993665367.


[I 2026-07-14 15:30:14,003] Trial 46 finished with value: 0.4776864292768341 and parameters: {'n_estimators': 169, 'max_depth': 6, 'learning_rate': 0.0877397090386724, 'subsample': 0.5717327815145163, 'colsample_bytree': 0.6417984020621186, 'reg_alpha': 0.9908803109591114, 'reg_lambda': 1.1250614571573417e-05, 'min_child_weight': 5}. Best is trial 33 with value: 0.46130134993665367.


[I 2026-07-14 15:30:14,927] Trial 47 finished with value: 0.46481910783464575 and parameters: {'n_estimators': 331, 'max_depth': 4, 'learning_rate': 0.03562469008906926, 'subsample': 0.5219073988170468, 'colsample_bytree': 0.5324957250381046, 'reg_alpha': 9.103000926887981e-07, 'reg_lambda': 0.0002336258784402063, 'min_child_weight': 3}. Best is trial 33 with value: 0.46130134993665367.


[I 2026-07-14 15:30:17,763] Trial 48 finished with value: 0.5244607104380964 and parameters: {'n_estimators': 399, 'max_depth': 9, 'learning_rate': 0.06357992938614133, 'subsample': 0.5746542131797204, 'colsample_bytree': 0.5934644112729581, 'reg_alpha': 0.008579756305441173, 'reg_lambda': 1.8880887450161209e-06, 'min_child_weight': 4}. Best is trial 33 with value: 0.46130134993665367.


[I 2026-07-14 15:30:18,316] Trial 49 finished with value: 0.5184106262284552 and parameters: {'n_estimators': 129, 'max_depth': 5, 'learning_rate': 0.01364480038835259, 'subsample': 0.5055671608879201, 'colsample_bytree': 0.6769601986446183, 'reg_alpha': 0.056322949091997754, 'reg_lambda': 3.0006245267526954e-05, 'min_child_weight': 2}. Best is trial 33 with value: 0.46130134993665367.
Best val MSE: 0.46130134993665367
Best params: {'n_estimators': 306, 'max_depth': 4, 'learning_rate': 0.03872656495281224, 'subsample': 0.5390840337870416, 'colsample_bytree': 0.561693501886788, 'reg_alpha': 9.298982208447791, 'reg_lambda': 0.03451772479612021, 'min_child_weight': 3}


In [5]:
# Refit on the full training set (tuning-train + tuning-val) with the best params,
# evaluate once on the untouched test set
best_params = study.best_params | {'enable_categorical': True, 'tree_method': 'hist', 'random_state': 42}
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train_full, y_train_full)

test_preds = final_model.predict(X_test)
print('Test RMSE:', mean_squared_error(y_test, test_preds) ** 0.5)
print('Test R^2:', r2_score(y_test, test_preds))
print()
print('GLM baseline for comparison: Test-scale RMSE ~0.7635 (from ERROR_MEAN_SQUARE on train), R^2 = 0.418')

Test RMSE: 0.6798982704662174
Test R^2: 0.6394331526003039

GLM baseline for comparison: Test-scale RMSE ~0.7635 (from ERROR_MEAN_SQUARE on train), R^2 = 0.418


## 4. Wider search: more trials, LightGBM, Random Forest

Bumping XGBoost to the same trial budget as the new models for a fair three-way comparison at equal search effort. All three use Optuna's TPESampler (Bayesian, Tree-structured Parzen Estimator) -- same tuning method throughout, just different model families.

In [ ]:
import time

N_TRIALS = 200
results = {}

# --- XGBoost, re-run with bigger budget ---
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'enable_categorical': True,
        'tree_method': 'hist',
        'random_state': 42,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return mean_squared_error(y_val, model.predict(X_val))

t0 = time.time()
xgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'XGBoost tuning took {time.time() - t0:.1f}s, best val MSE: {xgb_study.best_value:.4f}')

xgb_best = xgb_study.best_params | {'enable_categorical': True, 'tree_method': 'hist', 'random_state': 42}
xgb_final = xgb.XGBRegressor(**xgb_best)
xgb_final.fit(X_train_full, y_train_full)
xgb_test_preds = xgb_final.predict(X_test)
results['XGBoost'] = {
    'rmse': mean_squared_error(y_test, xgb_test_preds) ** 0.5,
    'r2': r2_score(y_test, xgb_test_preds),
    'best_params': xgb_study.best_params,
}
print(results['XGBoost'])

In [ ]:
import lightgbm as lgb

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'random_state': 42,
        'verbosity': -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
    return mean_squared_error(y_val, model.predict(X_val))

t0 = time.time()
lgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'LightGBM tuning took {time.time() - t0:.1f}s, best val MSE: {lgb_study.best_value:.4f}')

lgb_best = lgb_study.best_params | {'random_state': 42, 'verbosity': -1}
lgb_final = lgb.LGBMRegressor(**lgb_best)
lgb_final.fit(X_train_full, y_train_full)
lgb_test_preds = lgb_final.predict(X_test)
results['LightGBM'] = {
    'rmse': mean_squared_error(y_test, lgb_test_preds) ** 0.5,
    'r2': r2_score(y_test, lgb_test_preds),
    'best_params': lgb_study.best_params,
}
print(results['LightGBM'])

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# RandomForestRegressor needs numeric input -- one-hot encode categoricals
# (fit on train only, align test's columns to train's in case a category
# only appears in one split)
X_train_full_ohe = pd.get_dummies(X_train_full, columns=categorical_cols)
X_test_ohe = pd.get_dummies(X_test, columns=categorical_cols)
X_test_ohe = X_test_ohe.reindex(columns=X_train_full_ohe.columns, fill_value=0)

X_tr_ohe = X_train_full_ohe.iloc[:len(X_tr)]
X_val_ohe = X_train_full_ohe.iloc[len(X_tr):]

# NOTE: first attempt used the same N_TRIALS=200 with n_estimators up to 800 and
# max_depth up to 30 -- timed out after 40 minutes on a single nbconvert cell
# (deep, wide RF forests are much more expensive to fit than boosted shallow trees).
# Reduced to a smaller, more tractable search space and its own trial budget below.
RF_N_TRIALS = 60

def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 40),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0),
        'random_state': 42,
        'n_jobs': -1,
    }
    model = RandomForestRegressor(**params)
    model.fit(X_tr_ohe, y_tr)
    return mean_squared_error(y_val, model.predict(X_val_ohe))

t0 = time.time()
rf_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
rf_study.optimize(rf_objective, n_trials=RF_N_TRIALS, show_progress_bar=True)
print(f'Random Forest tuning took {time.time() - t0:.1f}s, best val MSE: {rf_study.best_value:.4f}')

rf_best = rf_study.best_params | {'random_state': 42, 'n_jobs': -1}
rf_final = RandomForestRegressor(**rf_best)
rf_final.fit(X_train_full_ohe, y_train_full)
rf_test_preds = rf_final.predict(X_test_ohe)
results['RandomForest'] = {
    'rmse': mean_squared_error(y_test, rf_test_preds) ** 0.5,
    'r2': r2_score(y_test, rf_test_preds),
    'best_params': rf_study.best_params,
}
print(results['RandomForest'])

In [ ]:
print(f'{"Model":<15} {"Test RMSE":<12} {"Test R^2":<10}')
print(f'{"GLM (baseline)":<15} {0.699:<12} {0.619:<10}')
for name, r in results.items():
    print(f'{name:<15} {r["rmse"]:<12.4f} {r["r2"]:<10.4f}')

**Result** (run as a standalone script rather than executed in-notebook, after the
nbconvert timeout on the original wide RF search space — same code as above, verified
output):

```
=== XGBoost ===
XGBoost tuning took 531.6s, best val MSE: 0.4578
{'rmse': 0.6766, 'r2': 0.6430, 'best_params': {'n_estimators': 257, 'max_depth': 4,
 'learning_rate': 0.0388, 'subsample': 0.552, 'colsample_bytree': 0.899,
 'reg_alpha': 0.00243, 'reg_lambda': 2.04e-06, 'min_child_weight': 18}}

=== LightGBM ===
LightGBM tuning took 79.7s, best val MSE: 0.4579
{'rmse': 0.6794, 'r2': 0.6399, 'best_params': {'n_estimators': 291, 'num_leaves': 113,
 'max_depth': 3, 'learning_rate': 0.1215, 'subsample': 0.764, 'colsample_bytree': 0.908,
 'reg_alpha': 0.00156, 'reg_lambda': 3.15e-07, 'min_child_samples': 57}}

=== Random Forest (reduced search space, 60 trials) ===
RandomForest tuning took 264.0s, best val MSE: 0.4563
{'rmse': 0.6680, 'r2': 0.6519, 'best_params': {'n_estimators': 111, 'max_depth': 11,
 'min_samples_split': 27, 'min_samples_leaf': 9, 'max_features': 0.634}}
```

**Final comparison:**

| Model | Test RMSE | Test R^2 |
|---|---|---|
| GLM (baseline, `delivery_delay_oml4py.ipynb`) | 0.699 | 0.619 |
| XGBoost (200 trials) | 0.677 | 0.643 |
| LightGBM (200 trials) | 0.679 | 0.640 |
| **Random Forest (60 trials, reduced search space)** | **0.668** | **0.652** |

Random Forest wins, despite the smallest search budget and simplest model family of
the three tree-based options -- genuinely interesting, not something to have predicted
in advance. All three tree-based models meaningfully beat GLM (real, apples-to-apples
comparison, same split/features/target-transform throughout), consistent with GLM's
structural inability to capture nonlinear interactions (distance x category, weight x
freight, etc.) that the checkout-time framing's features plausibly contain.

**Practical note on runtime:** the original attempt used the same 200-trial budget
and a wide search space (n_estimators up to 800, max_depth up to 30) for RF and timed
out after 40 minutes on a single cell -- RF's per-fit cost scales much faster with
depth/tree count than boosted models' shallow trees. Reducing to a tighter, more
sensible range and fewer trials still found a winning configuration, which is itself
a useful lesson: more trials/wider ranges isn't automatically better if it means the
search can't actually complete.